In [ ]:
import torch
import os
from torch.utils.data import DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import StepLR
import sys
sys.path.append('../')

import torch.nn.functional as F
from tqdm import tqdm
from  scripts.networks.unet import UNetModel
import matplotlib.pyplot as plt
from scripts.datasets import Dataset
from torch.utils.data import DataLoader
def highpass_filter_torch(freq, wavelet, dt, pad=[125,125], order=8):
    """
    Differentiable Butterworth high-pass filter (zero-phase),
    with reflect padding to suppress boundary artifacts.

    Args:
        freq (float): Cut-off frequency (Hz)
        wavelet (torch.Tensor): (..., nt)
        dt (float): Sampling interval (s)
        pad (tuple or None): (left, right) padding length
        order (int): Butterworth order

    Returns:
        torch.Tensor: Filtered wavelet
    """

    if pad is not None:
       
        wavelet = F.pad(wavelet, pad, mode="reflect")

    nt = wavelet.shape[-1]
    device = wavelet.device

    freqs = torch.fft.fftfreq(nt, d=dt).to(device).abs()

    eps = 1e-12
    H = 1.0 / torch.sqrt(
        1.0 + (freq / (freqs + eps)) ** (2 * order)
    )

    W = torch.fft.fft(wavelet, dim=-1)
    Wf = W * H
    wavelet = torch.fft.ifft(Wf, dim=-1).real

    if pad is not None:
        left, right = pad[0],pad[1]
        if right > 0:
            wavelet = wavelet[..., left:-right]
        else:
            wavelet = wavelet[..., left:]

    return wavelet

def plot_formal(tensor,savepath,savename,dx=10,dz=0.02):

    data = tensor[...,:-12].detach().cpu().numpy().squeeze()
    nz, nx = data.shape 
    
    extent = [0, nx * dx, nz * dz, 0]  # [x_min, x_max, y_min, y_max]
    llx1=0.2;lly1=0.05;llx2=0.01;lly2=0.1;ddx=1-llx1-llx2;ddy=1-lly1-lly2;

    fig, ax = plt.subplots(figsize=(1*3, 1*5*(ddx/ddy)))
    im = ax.matshow(data, cmap="gray", aspect="auto", origin="upper", extent=extent,vmin=-0.05*2,vmax=0.05*2)
    ax.set_position([llx1, lly1, ddx, ddy])

    x_ticks = range(0, nx, 100)  
    y_ticks = range(0, nz, 100)

    
    x_labels = [1 * i for i in x_ticks]   # keep 2 decimals
    y_labels = [round(i * dz, 3) for i in y_ticks]   # keep 3 decimals

    
   
    ax.set_xticks([i * dx for i in x_ticks])
    ax.set_xticklabels(x_labels)

    ax.set_yticks([i * dz for i in y_ticks])
    ax.set_yticklabels(y_labels)

    ax.set_xlabel("Trace")
    ax.set_ylabel("Time (s)")

    

    ax.xaxis.set_label_position('top')
    ax.xaxis.tick_top()
    
    plt.savefig(savepath + '/' + savename, dpi=600)
    plt.close(fig)
    
def show_batch_row(tensor, savepath  , savename,  cmap="gray"):

    if tensor.dim() == 4:
        tensor = tensor[:, 0, :, :]  
    if tensor.dim() == 5:
        tensor = tensor[:,0, 0, :, :]  

    
    b, h, w = tensor.shape
    data = tensor.detach().cpu().numpy()
    fig, axes = plt.subplots(1, b, figsize=(3*b, 3))
    if b == 1:
        axes = [axes]
    
    for i in range(b):
        axes[i].imshow(data[i], cmap='gray', aspect="auto", origin="upper",vmin=-0.05*2,vmax=0.05*2)
        axes[i].axis("off")

    plt.savefig(savepath + '/' + savename, dpi=600)
    plt.close(fig)

In [ ]:


batch_size = 1



use_cfg = False
device = 'cuda'
checkpoint_path = None  # or '/path/to/miniunet_50.pth'
ODE_step = 100
start_time = 30
option = 'post_fwi'
save_path = '../results/' +option 

def print_config(**kwargs):
    width = max(len(k) for k in kwargs) + 2
    print("\n" + "=" * 50)
    print(" Training Configuration ".center(50, "="))
    print("=" * 50)
    for k, v in kwargs.items():
        print(f"{k:<{width}}: {v}")
    print("=" * 50 + "\n")


print_config(

    batch_size=batch_size,
    save_path=save_path,
    use_cfg=use_cfg,
    device=device,
    checkpoint_path=checkpoint_path
)
os.makedirs(save_path, exist_ok=True)

In [ ]:
anno_path='../scripts/split_files'

test_anno='CGG_fwi_more.txt'



test_anno=os.path.join(anno_path, test_anno)


test_dataset=Dataset(
        test_anno,
        preload=True,
        lines=1,
        file_size=batch_size,
    )


test_dataloader = DataLoader(test_dataset, num_workers=1, batch_size=batch_size, shuffle=False)   


In [ ]:
model = UNetModel(
    attention_resolutions=[32, 16, 8],   # no need for 64 at 256px
    channel_mult=(1, 2, 4, 4, 8),           # balanced growth, good capacity
    in_channels=1,                       # grayscale input
    out_channels=1,                      # grayscale output
    use_scale_shift_norm=True,
    dropout=0.0,                         # keep dropout 0 for generation
    image_size=256,
    model_channels=128,                  # 128 base channels (lighter & stable)
    num_head_channels=64,                # 64 per attention head is standard
    num_res_blocks=2,                    # 2 resblocks per level = good balance
    resblock_updown=True
)

model.to(device)
model.eval()


# 加载RectifiedFlow

checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model'])
for param in model.parameters():
        param.requires_grad = False

In [ ]:

for iter, data in enumerate(test_dataloader):
        batch_bar = tqdm(range(start_time, ODE_step), desc="Generation steps", leave=False)        
        dt = 1.0 / ODE_step
        x_1 = data[0].to(device)

                        
        

        lr = 0.001  
        
        seed = 143
      
        g = torch.Generator(device=device).manual_seed(seed)   

        x_t = (start_time/ODE_step) * x_1 + (1- (start_time/ODE_step)) *torch.randn(x_1.shape, generator=g, device=device)


         
        latent = []


        loss_latent = []
        
        for index,j in enumerate(batch_bar):
                j = int(j)

                x_t = x_t.detach().clone().requires_grad_(True)
                
                optimizer = Adam([x_t], lr=lr)

                
                for epoch_i in range(1):
                        
                        optimizer.zero_grad()
                        
                        t = torch.tensor([j * dt], device=device)
                        v_pred = model(x=x_t, timesteps=t)
                        x11 = x_t + (1 - t) * v_pred

         
                        if j % 10 == 0:
                                latent.append(x11.detach().clone())
                        x11 = x11.clamp(-1, 1)

          

                 
                        loss = torch.nn.functional.mse_loss(x11, x_1, reduction='mean')

                        loss.backward()
                        optimizer.step()

                       
                        loss_latent.append(loss.item())

                        batch_bar.set_postfix({'Loss': f'{loss.item():.6f}'})
                                               
                v_pred = model(x=x_t, timesteps=t)
                        
                x_t = x_t + v_pred * dt

        generation_tensor = torch.stack(latent, dim=0)


In [ ]:
import numpy as np
plot_formal(x11,save_path,f'processed_fwi_{index}.png') 
plot_formal(x_1,save_path,f'oringal_fwi_{index}.png')   


show_batch_row(generation_tensor,save_path,f'generation_process_{index}.png')   



